# Molecular double-factorized Hamiltonian simulation

Companion notebook for the paper. Reproduces:
- Table I (double-factorization benchmark: orbitals, DF terms, reconstruction error, alpha-bar)
- Figure `fig:three_panels` (`figures/plot_all_orders_PRL_errorbars.pdf`)
- Figure `fig:partitioning_strategy_results` (`figures/nmin_vs_molecule.pdf`, appendix)

Depends only on `ffsim`, `pyscf`, `numpy`, `scipy`, `matplotlib`, and the local `trotter_methods.py` (LTS / PS / U_lambda / OG product formulas) -- no private packages.


## Table I: double-factorization benchmark

In [ ]:
"""
Double-Factorization Benchmark — convención exacta de ffsim
============================================================

La reconstrucción correcta (extraída de to_molecular_hamiltonian) es:

    df_nr = df_ham.to_number_representation()

    V[p,q,r,s] = sum_kij  Z_k[i,j]
                           * U_k[p,i] * U_k*[q,i]
                           * U_k[r,j] * U_k*[s,j]

donde Z_k = df_nr.diag_coulomb_mats[k]  y  U_k = df_nr.orbital_rotations[k]

Equivalentemente, definiendo
    L_k[p,q] = sum_i  U_k[p,i] * U_k*[q,i] * sqrt|z_k[i,i]|  ...

pero es más limpio usar directamente el einsum de ffsim.

Métricas:
    ||V - V_tilde||_F  : Frobenius, debe ser ~1e-15 (reconstrucción exacta)
    alpha_bar          : mean_{k>=1}  ||Z_k||_F / ||Z_0||_F
                         usando Z_k de la number representation
"""

import numpy as np
from pyscf import gto, scf
import ffsim

# ══════════════════════════════════════════════════════════════
#  MOLÉCULAS
# ══════════════════════════════════════════════════════════════
molecules = {
    "H2": {
        "atom": """
H 0.0000 0.0000 -0.3700
H 0.0000 0.0000  0.3700
""",
        "basis": "sto-3g",
    },
    "LiH": {
        "atom": """
Li 0.0000 0.0000 0.0000
H  0.0000 0.0000 3.0100
""",
        "basis": "sto-3g",
    },
    "BeH2": {
        "atom": """
Be 0.0000 0.0000  0.0000
H  0.0000 0.0000  1.3264
H  0.0000 0.0000 -1.3264
""",
        "basis": "sto-3g",
    },
    "H2O": {
        "atom": """
O 0.0000  0.0000  0.0000
H 0.0000  0.7572  0.5865
H 0.0000 -0.7572  0.5865
""",
        "basis": "sto-3g",
    },
    "NH3": {
        "atom": """
N  0.0000  0.0000  0.1173
H  0.0000  0.9391 -0.2730
H  0.8136 -0.4696 -0.2730
H -0.8136 -0.4696 -0.2730
""",
        "basis": "sto-3g",
    },
}


# ══════════════════════════════════════════════════════════════
#  RECONSTRUCCIÓN EXACTA  (misma fórmula que to_molecular_hamiltonian)
# ══════════════════════════════════════════════════════════════
def reconstruct_two_body_tensor(df_ham) -> np.ndarray:
    """
    V[p,q,r,s] = einsum('kij, kpi,kqi, krj,ksj -> pqrs',
                         Z_k, U_k, U_k*, U_k, U_k*)

    Usa to_number_representation() igual que ffsim internamente.
    """
    df_nr = df_ham.to_number_representation()
    return np.einsum(
        "kij,kpi,kqi,krj,ksj->pqrs",
        df_nr.diag_coulomb_mats,
        df_nr.orbital_rotations,
        df_nr.orbital_rotations.conj(),
        df_nr.orbital_rotations,
        df_nr.orbital_rotations.conj(),
    )


# ══════════════════════════════════════════════════════════════
#  MÉTRICAS
# ══════════════════════════════════════════════════════════════
def frobenius_error(mol_hamiltonian, df_ham) -> float:
    """
    ||V_orig - V_tilde||_F

    Nota: para una DF exacta (sin compresión de rango) esto será ~1e-15.
    Si usas optimize=True o truncas autovalores pequeños será mayor.
    """
    V_orig  = mol_hamiltonian.two_body_tensor
    V_tilde = reconstruct_two_body_tensor(df_ham).real
    return float(np.linalg.norm(V_orig - V_tilde))


def compute_alpha_bar(df_ham) -> float:
    """
    alpha_k = ||Z_k||_F / ||Z_0||_F   para k >= 1  (number representation)
    alpha_bar = media sobre k = 1, ..., L-1

    Se usa la number representation porque ahí Z_k tiene la escala física
    correcta (en la z_representation los signos cambian pero la norma también).
    """
    df_nr  = df_ham.to_number_representation()
    Zs     = df_nr.diag_coulomb_mats          # shape (L, norb, norb)
    norms  = np.linalg.norm(Zs.reshape(len(Zs), -1), axis=1)  # ||Z_k||_F

    norm_0 = norms[0]
    if norm_0 == 0.0:
        raise ValueError("||Z_0||_F = 0; no se puede normalizar.")

    alphas = norms[1:] / norm_0
    return float(alphas.mean())


# ══════════════════════════════════════════════════════════════
#  BUCLE PRINCIPAL
# ══════════════════════════════════════════════════════════════
print(f"{'Molécula':<10} {'N_orb':>6} {'L':>5} {'||V-Vtilde||_F':>16} {'alpha_bar':>12}")
print("─" * 54)

table_rows = {}

for mol_name, cfg in molecules.items():
    mol = gto.Mole()
    mol.build(atom=cfg["atom"], basis=cfg["basis"], unit="Angstrom", verbose=0)

    mf              = scf.RHF(mol).run()
    mol_data        = ffsim.MolecularData.from_scf(mf)
    norb            = mol_data.norb
    mol_hamiltonian = mol_data.hamiltonian

    df_ham = ffsim.DoubleFactorizedHamiltonian.from_molecular_hamiltonian(
        mol_hamiltonian
    )

    # Número de términos DF (en number representation, igual que z_representation)
    df_nr = df_ham.to_number_representation()
    L     = len(df_nr.orbital_rotations)

    frob_err  = frobenius_error(mol_hamiltonian, df_ham)
    alpha_bar = compute_alpha_bar(df_ham)

    table_rows[mol_name] = {
        "norb":      norb,
        "L":         L,
        "frob_err":  frob_err,
        "alpha_bar": alpha_bar,
    }

    print(f"{mol_name:<10} {norb:>6} {L:>5} {frob_err:>16.4e} {alpha_bar:>12.4f}")


# ══════════════════════════════════════════════════════════════
#  FILAS LaTeX
# ══════════════════════════════════════════════════════════════
latex_names = {
    "H2":   r"H$_2$",
    "LiH":  r"LiH",
    "BeH2": r"BeH$_2$",
    "H2O":  r"H$_2$O",
    "NH3":  r"NH$_3$",
}

print("\n\n% ── Filas LaTeX ────────────────────────────────────────")
for mol_name, row in table_rows.items():
    name = latex_names[mol_name]
    print(
        f"{name:<12} & {row['norb']:>3} & {row['L']:>4} "
        f"& \\num{{{row['frob_err']:.2e}}} "
        f"& {row['alpha_bar']:.4f} \\\\"
    )

## Data generation: state-error sweep over molecules and methods

Computes the raw per-sample errors behind `fig:three_panels` and caches them to `simulation_results_multi.npz` (already included in this repo, so this cell is safe to skip -- it will resume/reuse nothing automatically, it always recomputes; delete the .npz first if you want a clean run).

In [ ]:
"""
Quantum Chemistry Trotter Simulation — Multiple Random Initial States
=====================================================================
Molecules : H2, LiH, BeH2, H2O, NH3
Time      : t = 1  (total),  dt = 1e-2  →  n_steps = 100
Reference : scipy.sparse.linalg.expm_multiply  (exact matrix exponentiation)

For each molecule, N_SAMPLES random initial states are drawn (Haar-rotated HF).
All 8 methods are evaluated for each sample. ALL raw error values are saved —
no aggregation is done here; process them however you like afterwards.

Saved arrays per molecule (each shape = (N_SAMPLES,)):
    err_o0_trotter, err_o0_strang, err_o0_lam1, err_o0_lam2
    err_o1_trotter, err_o1_strang, err_o1_lam1, err_o1_lam_opt

Results saved to: simulation_results_multi.npz
Load with:
    data    = np.load("simulation_results_multi.npz", allow_pickle=True)
    results = data["results"].item()
    meta    = data["meta"].item()

Example — access raw errors for H2, order-0 Trotter:
    results["H2"]["err_o0_trotter"]   # numpy array of shape (N_SAMPLES,)
"""

import numpy as np
import scipy.sparse.linalg
from pyscf import gto, scf
import ffsim
from trotter_methods import (
    TrotterHamiltonian,
    simulate_trotter_double_factorized_PS,
    simulate_trotter_double_factorized_U_lambda,
    simulate_trotter_double_factorized_OG,
)

# ══════════════════════════════════════════════════════════════
#  USER PARAMETERS  — edit freely
# ══════════════════════════════════════════════════════════════
TIME_FINAL  = 1.0
DT          = 1e-2
N_STEPS_SIM = int(round(TIME_FINAL / DT))   # = 100

N_SAMPLES   = 10    # ← number of random initial states per molecule

LAMBDA_1   = 0.211325
LAMBDA_2   = 0.193183
LAMBDA_OPT = 0.211325

OUTPUT_FILE = "simulation_results_multi.npz"
MASTER_SEED = 42   # reproducibility; each (molecule, sample) gets a unique derived seed

print(f"Settings: t={TIME_FINAL}, dt={DT}, n_steps={N_STEPS_SIM}, N_SAMPLES={N_SAMPLES}")
print(f"  λ1={LAMBDA_1}, λ2={LAMBDA_2}, λ_opt={LAMBDA_OPT}")
print(f"  MASTER_SEED={MASTER_SEED}")
print(f"  Output: {OUTPUT_FILE}\n")

# ══════════════════════════════════════════════════════════════
#  MOLECULE DEFINITIONS
# ══════════════════════════════════════════════════════════════
molecules = {
    "H2": {
        "atom": """
H 0.0000 0.0000 -0.3700
H 0.0000 0.0000  0.3700
""",
        "basis": "sto-3g",
    },
    "LiH": {
        "atom": """
Li 0.0000 0.0000 0.0000
H  0.0000 0.0000 3.0100
""",
        "basis": "sto-3g",
    },
    "BeH2": {
        "atom": """
Be 0.0000 0.0000  0.0000
H  0.0000 0.0000  1.3264
H  0.0000 0.0000 -1.3264
""",
        "basis": "sto-3g",
    },
    "H2O": {
        "atom": """
O 0.0000  0.0000  0.0000
H 0.0000  0.7572  0.5865
H 0.0000 -0.7572  0.5865
""",
        "basis": "sto-3g",
    },
    "NH3": {
        "atom": """
N  0.0000  0.0000  0.1173
H  0.0000  0.9391 -0.2730
H  0.8136 -0.4696 -0.2730
H -0.8136 -0.4696 -0.2730
""",
        "basis": "sto-3g",
    },
}

# ══════════════════════════════════════════════════════════════
#  RANDOM STATE GENERATOR
# ══════════════════════════════════════════════════════════════
def make_random_state(norb: int, nelec: tuple, rng: np.random.Generator) -> np.ndarray:
    """
    Return a random normalized state in the (N_alpha, N_beta) Fock sector.

    Applies a Haar-random orbital rotation U ∈ U(norb) to the HF state,
    which stays inside the correct symmetry sector without any projection.

    Args:
        norb : number of spatial orbitals
        nelec: (n_alpha, n_beta) electron numbers
        rng  : numpy random Generator

    Returns:
        Normalized complex state vector in the (N_alpha, N_beta) sector.
    """
    hf_state = ffsim.hartree_fock_state(norb, nelec)

    # Random unitary from the Haar measure via QR decomposition
    Z = rng.standard_normal((norb, norb)) + 1j * rng.standard_normal((norb, norb))
    Q, R = np.linalg.qr(Z)
    phase = np.diag(R) / np.abs(np.diag(R))
    random_unitary = Q * phase[np.newaxis, :]

    random_state = ffsim.apply_orbital_rotation(
        hf_state, random_unitary, norb=norb, nelec=nelec
    )

    norm = np.linalg.norm(random_state)
    return random_state / norm


# ══════════════════════════════════════════════════════════════
#  PER-MOLECULE SIMULATION
# ══════════════════════════════════════════════════════════════
def process_molecule(atom_string: str, basis: str, mol_name: str, mol_index: int) -> dict:
    # ── Build PySCF molecule and Hamiltonians ────────────────
    mol = gto.Mole()
    mol.build(atom=atom_string, basis=basis, unit="Angstrom", verbose=0)

    mf              = scf.RHF(mol).run()
    mol_data        = ffsim.MolecularData.from_scf(mf)
    norb            = mol_data.norb
    nelec           = mol_data.nelec
    mol_hamiltonian = mol_data.hamiltonian

    df_hamiltonian           = ffsim.DoubleFactorizedHamiltonian.from_molecular_hamiltonian(
                                   mol_hamiltonian)
    hamiltonian_uncompressed = TrotterHamiltonian.from_ffsim(df_hamiltonian)

    # ── Pre-build the linear operator for exact reference ───
    linop   = ffsim.linear_operator(mol_hamiltonian, norb=norb, nelec=nelec)
    trace_H = ffsim.trace(mol_hamiltonian, norb=norb, nelec=nelec)

    # ── Hamiltonian energy errors (independent of initial state) ──
    # optimize=False reconstructs the Hamiltonian exactly, so these should
    # be at machine precision; computed directly from ffsim, no custom class.
    recon = df_hamiltonian.to_molecular_hamiltonian()
    err_one = float(np.linalg.norm(recon.one_body_tensor - mol_hamiltonian.one_body_tensor))
    err_two = float(np.linalg.norm(recon.two_body_tensor - mol_hamiltonian.two_body_tensor))

    # ── Accumulators — one list per method ──────────────────
    raw = {
        "err_o0_trotter":  [],
        "err_o0_strang":   [],
        "err_o0_lam1":     [],
        "err_o0_lam2":     [],
        "err_o1_trotter":  [],
        "err_o1_strang":   [],
        "err_o1_lam1":     [],
        "err_o1_lam_opt":  [],
    }

    # ── Loop over N_SAMPLES random initial states ────────────
    for sample_idx in range(N_SAMPLES):
        # Unique, reproducible seed for this (molecule, sample) pair
        seed = MASTER_SEED * 10_000 + mol_index * 1_000 + sample_idx
        rng  = np.random.default_rng(seed)

        initial_state = make_random_state(norb, nelec, rng)

        # Reference: exact time evolution
        exact_state = scipy.sparse.linalg.expm_multiply(
            -1j * TIME_FINAL * linop,
            initial_state,
            traceA=-1j * TIME_FINAL * trace_H,
        )

        # ── ORDER 0 ─────────────────────────────────────────

        s = ffsim.simulate_trotter_double_factorized(
            initial_state, df_hamiltonian, TIME_FINAL,
            norb=norb, nelec=nelec, n_steps=N_STEPS_SIM, order=0,
        )
        raw["err_o0_trotter"].append(float(np.linalg.norm(s - exact_state)))

        s = simulate_trotter_double_factorized_PS(
            initial_state, hamiltonian_uncompressed, TIME_FINAL,
            norb=norb, nelec=nelec, n_steps=N_STEPS_SIM, order=0,
        )
        raw["err_o0_strang"].append(float(np.linalg.norm(s - exact_state)))

        s = simulate_trotter_double_factorized_U_lambda(
            initial_state, hamiltonian_uncompressed, TIME_FINAL,
            Lambda=LAMBDA_1, norb=norb, nelec=nelec, n_steps=N_STEPS_SIM, order=0,
        )
        raw["err_o0_lam1"].append(float(np.linalg.norm(s - exact_state)))

        s = simulate_trotter_double_factorized_U_lambda(
            initial_state, hamiltonian_uncompressed, TIME_FINAL,
            Lambda=LAMBDA_2, norb=norb, nelec=nelec, n_steps=N_STEPS_SIM, order=0,
        )
        raw["err_o0_lam2"].append(float(np.linalg.norm(s - exact_state)))

        # ── ORDER 1 ─────────────────────────────────────────

        s = ffsim.simulate_trotter_double_factorized(
            initial_state, df_hamiltonian, TIME_FINAL,
            norb=norb, nelec=nelec, n_steps=N_STEPS_SIM, order=1,
        )
        raw["err_o1_trotter"].append(float(np.linalg.norm(s - exact_state)))

        s = simulate_trotter_double_factorized_PS(
            initial_state, hamiltonian_uncompressed, TIME_FINAL,
            norb=norb, nelec=nelec, n_steps=N_STEPS_SIM, order=1,
        )
        raw["err_o1_strang"].append(float(np.linalg.norm(s - exact_state)))

        s = simulate_trotter_double_factorized_U_lambda(
            initial_state, hamiltonian_uncompressed, TIME_FINAL,
            Lambda=LAMBDA_1, norb=norb, nelec=nelec, n_steps=N_STEPS_SIM, order=1,
        )
        raw["err_o1_lam1"].append(float(np.linalg.norm(s - exact_state)))

        s = simulate_trotter_double_factorized_OG(
            initial_state, hamiltonian_uncompressed, TIME_FINAL,
            norb=norb, nelec=nelec, lambda_opt=LAMBDA_OPT, n_steps=N_STEPS_SIM,
        )
        raw["err_o1_lam_opt"].append(float(np.linalg.norm(s - exact_state)))

        print(f"    sample {sample_idx+1:>3}/{N_SAMPLES}  "
              f"o0_trotter={raw['err_o0_trotter'][-1]:.3e}  "
              f"o1_trotter={raw['err_o1_trotter'][-1]:.3e}")

    # Convert lists → numpy arrays (shape = N_SAMPLES,)
    for key in raw:
        raw[key] = np.array(raw[key])

    return {
        **raw,                          # raw error arrays, shape (N_SAMPLES,)
        "err_one_body": float(err_one),
        "err_two_body": float(err_two),
        "norb":         norb,
        "nelec":        nelec,
        "basis":        basis,
        "N_SAMPLES":    N_SAMPLES,
    }


# ══════════════════════════════════════════════════════════════
#  RUN ALL MOLECULES
# ══════════════════════════════════════════════════════════════
results = {}
for mol_index, (mol_name, mol_cfg) in enumerate(molecules.items()):
    print(f"\n→ Processing {mol_name}  (basis={mol_cfg['basis']}) ...")
    results[mol_name] = process_molecule(
        mol_cfg["atom"], mol_cfg["basis"], mol_name, mol_index
    )
    r = results[mol_name]
    # Quick summary (mean ± std) printed for sanity check — NOT saved
    for key in ["err_o0_trotter", "err_o0_strang", "err_o0_lam1", "err_o0_lam2",
                "err_o1_trotter", "err_o1_strang", "err_o1_lam1", "err_o1_lam_opt"]:
        arr = r[key]
        print(f"  {key:<20}  mean={arr.mean():.3e}  std={arr.std():.3e}  "
              f"min={arr.min():.3e}  max={arr.max():.3e}")


# ══════════════════════════════════════════════════════════════
#  SAVE TO NPZ  — raw arrays only, no aggregation
# ══════════════════════════════════════════════════════════════
meta = {
    "TIME_FINAL":   TIME_FINAL,
    "DT":           DT,
    "N_STEPS_SIM":  N_STEPS_SIM,
    "N_SAMPLES":    N_SAMPLES,
    "reference":    "expm_multiply (exact)",
    "LAMBDA_1":     LAMBDA_1,
    "LAMBDA_2":     LAMBDA_2,
    "LAMBDA_OPT":   LAMBDA_OPT,
    "MASTER_SEED":  MASTER_SEED,
    "mol_names":    list(results.keys()),
    # Seed formula for reproducibility:
    #   seed = MASTER_SEED * 10_000 + mol_index * 1_000 + sample_idx
}

np.savez(OUTPUT_FILE, results=results, meta=meta)

print(f"\nResults saved → {OUTPUT_FILE}")
print("Load with:")
print("  data    = np.load('simulation_results_multi.npz', allow_pickle=True)")
print("  results = data['results'].item()")
print("  meta    = data['meta'].item()")
print()
print("Each results[mol_name][method_key] is a numpy array of shape (N_SAMPLES,).")
print("Keys per molecule:")
print("  err_o0_trotter, err_o0_strang, err_o0_lam1, err_o0_lam2")
print("  err_o1_trotter, err_o1_strang, err_o1_lam1, err_o1_lam_opt")
print("  err_one_body, err_two_body, norb, nelec, basis, N_SAMPLES")

## Figure: `fig:three_panels` (`plot_all_orders_PRL_errorbars.pdf`)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from matplotlib.lines import Line2D
from matplotlib.ticker import LogLocator, LogFormatterMathtext

plt.rcParams.update({
    "font.size": 25,
    "axes.labelsize": 20,
    "legend.fontsize": 25,
    "xtick.labelsize": 17,
    "ytick.labelsize": 17,
    "font.family": "serif",
    "text.usetex":         False,
    "axes.linewidth":       0.8,
    "xtick.direction":     "in",
    "ytick.direction":     "in",
    "xtick.top":           True,
    "ytick.right":         True,
    "xtick.minor.visible": True,
    "ytick.minor.visible": True,
})


# ══════════════════════════════════════════════════════════════
#  LOAD
# ══════════════════════════════════════════════════════════════
data    = np.load("simulation_results_multi.npz", allow_pickle=True)
results = data["results"].item()
meta    = data["meta"].item()

TIME_FINAL  = meta["TIME_FINAL"]
DT          = meta["DT"]
N_STEPS_SIM = meta["N_STEPS_SIM"]
LAMBDA_1    = meta["LAMBDA_1"]
LAMBDA_2    = meta["LAMBDA_2"]
LAMBDA_OPT  = meta["LAMBDA_OPT"]
mol_names   = meta["mol_names"]
N_SAMPLES   = meta["N_SAMPLES"]

_fixed_order = [m for m in mol_names if m not in ("NH3", "H2O")] + ["NH3", "H2O"]
mol_names    = [m for m in _fixed_order if m in mol_names]

print(f"Loaded: {mol_names}")
print(f"t={TIME_FINAL}, Δt={DT}, n_steps={N_STEPS_SIM}, N_SAMPLES={N_SAMPLES}")
print(f"λ1={LAMBDA_1}, λ2={LAMBDA_2}, λ_opt={LAMBDA_OPT}\n")

# ══════════════════════════════════════════════════════════════
#  ETIQUETAS MOLECULAS
# ══════════════════════════════════════════════════════════════
molecules = ['$H_2$', '$LiH$', '$BeH_2$', '$NH_3$', '$H_2O$']
#mol_display = [mol_name_labels.get(m, m) for m in mol_names]

# ══════════════════════════════════════════════════════════════
#  MEDIA Y STD
# ══════════════════════════════════════════════════════════════
def mean_std(key):
    vals = np.array([results[m][key] for m in mol_names])
    return vals.mean(axis=1), vals.std(axis=1)

e_o0_trotter, s_o0_trotter = mean_std("err_o0_trotter")
e_o0_strang,  s_o0_strang  = mean_std("err_o0_strang")
e_o0_lam1,    s_o0_lam1    = mean_std("err_o0_lam1")

e_o1_trotter, s_o1_trotter = mean_std("err_o1_trotter")
e_o1_strang,  s_o1_strang  = mean_std("err_o1_strang")
e_o1_lam1,    s_o1_lam1    = mean_std("err_o1_lam1")
e_o1_lam_opt, s_o1_lam_opt = mean_std("err_o1_lam_opt")

x_pos = np.arange(len(mol_names))

# ══════════════════════════════════════════════════════════════
#  ESTILOS
# ══════════════════════════════════════════════════════════════


STYLES_O0 = {
    "trotter": dict(color='black',   ls='-',  marker='o', lw=2.0, ms=6, mfc='black',   order=1),
    "strang":  dict(color='#d62728', ls='-',  marker='s', lw=2.0, ms=6, mfc='#d62728', order=2),
    "lam1":    dict(color='#2ca02c', ls='-',  marker='D', lw=1.8, ms=5, mfc='#2ca02c', order=4),
}
STYLES_O1 = {
    "trotter": dict(color='black',   ls='--', marker='o', lw=1.7, ms=5, mfc='black',   order=5),
    "strang":  dict(color='#d62728', ls='--', marker='s', lw=1.7, ms=5, mfc='white',   order=6),
    "lam1":    dict(color='#ff7f0e', ls='--', marker='v', lw=1.7, ms=7, mfc='white',   order=8),
    "lam_opt": dict(color='#17becf', ls='--', marker='v', lw=1.7, ms=10, mfc='#17becf',   order=9),
}

LABELS_O0 = {
    "trotter": r"LTS$^{[1]}$",
    "strang":  r"PS$^{[1]}$",
    "lam1":    r"$U_{\tilde{\lambda}}^{[1]}$",
}
LABELS_O1 = {
    "trotter": r"LTS$^{[2]}$",
    "strang":  r"PS$^{[2]}$",
    "lam1":    r"$U_{\tilde{\lambda}}^{[2]}$",
    "lam_opt": r"OG$^{[2]}$",
}

# ══════════════════════════════════════════════════════════════
#  HELPERS
# ══════════════════════════════════════════════════════════════
def plot_curves_with_errorbars(ax, x, datasets, errors, styles, labels,
                                draw_order, capsize=2.5, elinewidth=0.8,
                                alpha_eb=0.55):
    for z, key in enumerate(draw_order, start=2):
        mean = datasets[key]
        std  = errors[key]
        st   = styles[key]
        lbl  = labels[key]
        ax.errorbar(
            x, mean, yerr=std,
            fmt="none", ecolor=st["color"],
            elinewidth=elinewidth, capsize=capsize,
            capthick=elinewidth, alpha=alpha_eb, zorder=z,
        )
        ax.plot(
            x, mean,
            color=st["color"], linewidth=st["lw"],
            linestyle=st["ls"], marker=st["marker"],
            markersize=st["ms"], markerfacecolor=st["mfc"],
            markeredgecolor=st["color"], markeredgewidth=0.8,
            label=lbl, zorder=z + 0.5,
        )

def make_handle(st):
    return Line2D([], [],
                  color=st["color"], linewidth=st["lw"],
                  linestyle=st["ls"], marker=st["marker"],
                  markersize=st["ms"], markerfacecolor=st["mfc"],
                  markeredgecolor=st["color"])

# ══════════════════════════════════════════════════════════════
#  FIGURA
# ══════════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(7, 5))

# Order 0
plot_curves_with_errorbars(
    ax, x_pos,
    datasets   = {"trotter": e_o0_trotter, "strang": e_o0_strang,
                  "lam1": e_o0_lam1},
    errors     = {"trotter": s_o0_trotter, "strang": s_o0_strang,
                  "lam1": s_o0_lam1},
    styles=STYLES_O0, labels=LABELS_O0,
    draw_order=["lam1", "strang", "trotter"],
)

# Order 1
plot_curves_with_errorbars(
    ax, x_pos,
    datasets   = {"trotter": e_o1_trotter, "strang": e_o1_strang,
                  "lam1": e_o1_lam1, "lam_opt": e_o1_lam_opt},
    errors     = {"trotter": s_o1_trotter, "strang": s_o1_strang,
                  "lam1": s_o1_lam1, "lam_opt": s_o1_lam_opt},
    styles=STYLES_O1, labels=LABELS_O1,
    draw_order=["lam_opt", "lam1", "strang", "trotter"],
)

ax.set_yscale("log")
ax.set_xlabel("Molecule")
ax.set_ylabel(r'$\|U - \tilde{U}\|$')
ax.set_xticks(x_pos)
ax.set_xticklabels(molecules)
ax.tick_params(axis='x', which='both', top=False, bottom=True, length=4)
ax.xaxis.set_minor_locator(ticker.NullLocator())
ax.tick_params(which='major', length=4, width=0.7)
ax.tick_params(which='minor', length=2, width=0.5)
ax.grid(False)

# números eje Y a la derecha, etiqueta a la izquierda
ax.yaxis.tick_right()
ax.yaxis.set_label_position('left')
ax.yaxis.set_major_locator(LogLocator(base=10.0, numticks=6))
ax.yaxis.set_major_formatter(LogFormatterMathtext())
ax.yaxis.set_minor_locator(LogLocator(base=10.0, subs=np.arange(2, 10), numticks=50))
ax.yaxis.set_minor_formatter(ticker.NullFormatter())

# texto Delta t
ax.text(0.05, 0.97, rf'$\Delta t={DT}$',
        transform=ax.transAxes, fontsize=11,
        va='top', ha='left',
        bbox=dict(boxstyle='square,pad=0.3', fc='white', ec='black', lw=0.6))

# leyenda
h_o0 = [make_handle(STYLES_O0[k]) for k in ["trotter", "strang", "lam1"]]
l_o0 = [LABELS_O0[k]              for k in ["trotter", "strang", "lam1"]]
h_o1 = [make_handle(STYLES_O1[k]) for k in ["trotter", "strang", "lam1", "lam_opt"]]
l_o1 = [LABELS_O1[k]              for k in ["trotter", "strang", "lam1", "lam_opt"]]

leg = ax.legend(
    h_o0 + h_o1,
    l_o0 + l_o1,
    frameon=True, framealpha=1.0,
    edgecolor='black', fancybox=False,
    fontsize=11, ncol=2, loc='lower right',
    handlelength=2.2, handletextpad=0.4,
    labelspacing=0.35, borderpad=0.5,
)
leg.get_frame().set_linewidth(0.6)

plt.tight_layout(pad=0.8)
plt.savefig("figures/plot_all_orders_PRL_errorbars.pdf", dpi=300)
plt.savefig("figures/plot_all_orders_PRL_errorbars.png", dpi=300)
plt.show()
print("Saved → plot_all_orders_PRL_errorbars.pdf / .png")

## Figure (appendix): `fig:partitioning_strategy_results` (`nmin_vs_molecule.pdf`)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np

plt.rcParams.update({
    "font.size":        25,
    "axes.labelsize":   20,
    "legend.fontsize":  28,
    "xtick.labelsize":  17,
    "ytick.labelsize":  17,
    "font.family":      "serif",
    "text.usetex":      False,
    "axes.linewidth":   0.8,
    "xtick.direction":  "in",
    "ytick.direction":  "in",
    "xtick.top":        True,
    "ytick.right":      True,
    "xtick.minor.visible": True,
    "ytick.minor.visible": True,
})

molecules = ['$H_2$', '$LiH$', '$BeH_2$', '$NH_3$', '$H_2O$']
x = np.arange(len(molecules))

data = {
    'SI / LTS':  [26,  123, 123,  518, 814],
    'SI / PS':   [2,   13,  21,   112, 246],
    'MCC / LTS': [26,  98,  236,  657, 950],
    'MCC / PS':  [2,   107, 234,  651, 934],
}

styles = {
    'SI / LTS':  {'color': 'black',   'linestyle': '-',  'marker': 'o'},
    'SI / PS':   {'color': 'black',   'linestyle': '--', 'marker': 's'},
    'MCC / LTS': {'color': '#c0392b', 'linestyle': '-',  'marker': 'o'},
    'MCC / PS':  {'color': '#c0392b', 'linestyle': '--', 'marker': 's'},
}

legend_labels = {
    'SI / LTS':  r'SI / $LTS^{[1]}$',
    'SI / PS':   r'SI / $PS^{[1]}$',
    'MCC / LTS': r'MCC / $LTS^{[1]}$',
    'MCC / PS':  r'MCC / $PS^{[1]}$',
}

fig, ax = plt.subplots(figsize=(7, 5))

for label, values in data.items():
    s = styles[label]
    ax.plot(x, values,
            color=s['color'],
            linestyle=s['linestyle'],
            marker=s['marker'],
            markersize=8,
            linewidth=1.8,
            label=legend_labels[label])  # ← usa legend_labels, no label

ax.xaxis.set_major_locator(ticker.FixedLocator(x))
ax.xaxis.set_major_formatter(ticker.FixedFormatter(molecules))
ax.xaxis.set_minor_locator(ticker.NullLocator())
ax.set_xlabel("Molecule")
ax.set_ylabel(r'$N_{\min} (\|U - \tilde{U}\| < \epsilon)$')

ax.yaxis.set_major_formatter(ticker.LogFormatterSciNotation(base=10, labelOnlyBase=True))
ax.yaxis.set_major_locator(ticker.LogLocator(base=10, numticks=5))
ax.yaxis.tick_right()
ax.yaxis.set_label_position('left')
ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{x:g}'))
ax.yaxis.set_minor_formatter(ticker.FuncFormatter(lambda x, _: f'{x:g}'))

ax.set_ylim(0.8, 2000)
ax.set_yscale('log')

ax.legend(fontsize=11, framealpha=0.9, edgecolor='black')
plt.tight_layout()
plt.savefig('figures/nmin_vs_molecule.pdf', dpi=300, bbox_inches='tight')
plt.show()